# Engramma Memory — 5-Minute Quickstart

This notebook demonstrates the core Engramma API: **store**, **query**, **compose**, **retrieve**, and **forget**.

By the end, you'll understand why Engramma is a *memory engine*, not just a vector store.

In [ ]:
# Install if needed
# !pip install engramma-memory

In [ ]:
import numpy as np
from engramma_memory import EngrammaMemory

# Create a memory engine (64-dimensional, local backend)
mem = EngrammaMemory(dim=64, backend="local")
print(mem)

## 1. Store Embeddings

In practice, these would come from an embedding model (OpenAI, sentence-transformers, etc.).
Here we generate deterministic random vectors for demonstration.

In [ ]:
rng = np.random.default_rng(42)

concepts = {}
for name in ["python", "javascript", "machine_learning", "web_dev", "data_science"]:
    embedding = rng.standard_normal(64).astype(np.float32)
    embedding /= np.linalg.norm(embedding)
    mem.store(key=embedding, value=embedding)
    concepts[name] = embedding
    print(f"Stored: {name} (norm={np.linalg.norm(embedding):.4f})")

print(f"\nTotal patterns: {mem.count}")

## 2. Query — Find Relevant Memories

`query()` returns the top-k most similar patterns, each with a score.

In [ ]:
results = mem.query(concepts["python"], top_k=3)

print("Query: 'python'")
print(f"Results ({len(results)} returned):")
for i, r in enumerate(results):
    print(f"  #{i+1} score={r['score']:.4f}")

## 3. Compose — Blend Multiple Patterns (Engramma's Killer Feature)

This is what vector databases **cannot** do natively.

Engramma uses multi-head attention to compose stored patterns. Each head attends to different aspects of the input keys, producing a coherent blend — not a naive average.

In [ ]:
# Compose "python" + "data_science"
blend = mem.compose([concepts["python"], concepts["data_science"]])

print(f"Composed 'python' + 'data_science':")
print(f"  Vector norm: {np.linalg.norm(blend):.4f}")

# Compare: how similar is the blend to each component?
for name in ["python", "data_science", "javascript", "web_dev"]:
    sim = np.dot(blend.flatten()[:64], concepts[name]) / (
        np.linalg.norm(blend) * np.linalg.norm(concepts[name]) + 1e-8
    )
    print(f"  Similarity to '{name}': {sim:.4f}")

## 4. Retrieve — Smart Routing

The `ConfidenceRouter` automatically picks the best pathway:
- Exact Memory → for high-confidence matches
- Energy Memory → for soft generalization  
- Multi-Head Attention → for compositional patterns

In [ ]:
result = mem.retrieve(concepts["machine_learning"])

similarity = np.dot(result.flatten()[:64], concepts["machine_learning"]) / (
    np.linalg.norm(result) * np.linalg.norm(concepts["machine_learning"]) + 1e-8
)
print(f"Smart retrieve 'machine_learning':")
print(f"  Similarity to original: {similarity:.4f}")

## 5. Forget — Remove Patterns

In [ ]:
print(f"Before forget: {mem.count} patterns")

mem.forget(concepts["javascript"], strategy="immediate")
print(f"After forgetting 'javascript': {mem.count} patterns")

mem.forget(concepts["web_dev"], strategy="decay")
print(f"After decaying 'web_dev': {mem.count} patterns")

## 6. Stats

In [ ]:
stats = mem.stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

## Next Steps

- **[02_chatbot_memory.ipynb](02_chatbot_memory.ipynb)** — Build a chatbot with long-term memory
- **[03_composition_deep_dive.ipynb](03_composition_deep_dive.ipynb)** — Understand why composition beats kNN

---

### Ready for production?

The local backend caps at 1000 patterns. For unlimited storage, persistence, and weighted composition:

```python
mem = EngrammaMemory(dim=64, backend="cloud", api_key="nx_live_...")
```

Get your free key: [www.engramma-memory.com/signup](https://www.engramma-memory.com/signup)